# Layer Indexing

Explores exact labels, substring suggestions, and saved activation lookup behavior. Later intervention and sweep notebooks depend on these site lookup conventions.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.LayerLog.buffer_parent",
    "tl.LayerLog.captured_args",
    "tl.LayerLog.captured_kwargs",
    "tl.LayerLog.child_layers",
    "tl.LayerLog.child_layers_per_pass",
    "tl.LayerLog.child_passes_per_layer",
    "tl.LayerLog.co_parent_layers",
    "tl.LayerLog.cond_branch_children_by_cond",
    "tl.LayerLog.cond_branch_elif_children",
    "tl.LayerLog.cond_branch_else_children",
    "tl.LayerLog.cond_branch_start_children",
    "tl.LayerLog.cond_branch_then_children",
    "tl.LayerLog.conditional_branch_stack_passes",
    "tl.LayerLog.conditional_branch_stacks",
    "tl.LayerLog.containing_module",
    "tl.LayerLog.containing_modules",
    "tl.LayerLog.corresponding_grad_fn",
    "tl.LayerLog.creation_order",
    "tl.LayerLog.detach_saved_tensor",
    "tl.LayerLog.edges_vary_across_passes",
    "tl.LayerLog.equivalent_operations",
    "tl.LayerLog.extra_data",
    "tl.LayerLog.flops_backward",
    "tl.LayerLog.flops_forward",
    "tl.LayerLog.func_applied",
    "tl.LayerLog.func_argnames",
    "tl.LayerLog.func_call_stack",
    "tl.LayerLog.func_config",
    "tl.LayerLog.func_is_inplace",
    "tl.LayerLog.func_name",
    "tl.LayerLog.func_rng_states",
    "tl.LayerLog.func_time",
    "tl.LayerLog.get_child_layers",
    "tl.LayerLog.get_parent_layers",
    "tl.LayerLog.grad_fn_id",
    "tl.LayerLog.grad_fn_name",
    "tl.LayerLog.grad_fn_object",
    "tl.LayerLog.gradient",
    "tl.LayerLog.has_children",
    "tl.LayerLog.has_co_parents",
    "tl.LayerLog.has_gradient",
    "tl.LayerLog.has_parents",
    "tl.LayerLog.has_saved_activations",
    "tl.LayerLog.has_siblings",
    "tl.LayerLog.is_buffer_layer",
    "tl.LayerLog.is_computed_inside_submodule",
    "tl.LayerLog.is_final_output",
    "tl.LayerLog.is_input_layer",
    "tl.LayerLog.is_internally_initialized",
    "tl.LayerLog.is_internally_terminated",
    "tl.LayerLog.is_output_layer",
    "tl.LayerLog.is_part_of_iterable_output",
    "tl.LayerLog.is_scalar_bool",
    "tl.LayerLog.is_terminal_bool_layer",
    "tl.LayerLog.iterable_output_index",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")